# Investigate Exact Repeated Training Rows

This notebook examines the exact repeats found in the training period. It does not change the raw table or decide whether rows should be removed. A deduplicated copy is created only as a hypothetical comparison.

## Request in plain language

> Show me what the exact repeated rows are, how often they occur, which training groups they affect, and how a hypothetical collapse of exact repeats would change summary counts. Keep proof cancellation outcomes out of the analysis and do not alter the source data.

**Important:** identical rows can still represent separate bookings. The file has no booking identifier, so “exact repeat” does not prove “bad duplicate.”

## 1. Load the source and isolate train

Only the training period is examined here. The other periods are not used for duplicate analysis.

In [1]:
from pathlib import Path
import sys

repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "data.py").exists():
        repo_root = folder
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")

source_folder = str(repo_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)

import contract
import data
import pandas as pd

raw_bookings = data.read_raw()
bookings = raw_bookings.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
train, choose, proof = contract.split_periods(bookings)
source_columns = raw_bookings.columns.tolist()

print(f"Training rows: {len(train):,}")
print(f"Source columns used to identify exact repeats: {len(source_columns)}")

Training rows: 49,157
Source columns used to identify exact repeats: 32


## 2. Count exact repeated rows and groups

`duplicated()` counts later matching rows. A group of three identical rows contributes two later repeats. We also count how many groups exist and how many rows belong to those groups.

In [2]:
repeated_after_first = train.duplicated(subset=source_columns)
all_duplicate_members = train.duplicated(subset=source_columns, keep=False)

group_sizes = train[source_columns].value_counts(dropna=False)
duplicate_groups = group_sizes[group_sizes > 1]

print(f"Later matching rows: {int(repeated_after_first.sum()):,}")
print(f"Exact duplicate groups: {len(duplicate_groups):,}")
print(f"Rows belonging to duplicate groups: {int(duplicate_groups.sum()):,}")
print(f"Unique source rows if exact repeats were collapsed: {len(group_sizes):,}")

group_size_counts = duplicate_groups.value_counts().sort_index()
group_size_summary = group_size_counts.rename_axis("copies_in_group").rename("number_of_groups")
print("Duplicate group sizes:")
print(group_size_summary.to_string())

Later matching rows: 16,291
Exact duplicate groups: 3,472
Rows belonging to duplicate groups: 19,763
Unique source rows if exact repeats were collapsed: 32,866
Duplicate group sizes:
copies_in_group
2      1963
3       396
4       214
5       119
6        95
7        65
8        57
9        42
10       68
11       33
12       33
13       22
14       20
15       28
16       20
17       15
18       15
19       14
20       70
21        6
22        8
23        8
24       13
25       24
26        6
27        4
28        8
29       29
30       21
31        1
32        3
33        2
34        1
35        1
36        2
38        1
39        1
40        6
41        1
44        1
45        2
46        1
47        1
49        1
50        6
52        2
55        2
58        4
60        3
62        1
65        4
69        1
70        2
72        1
89        1
100       1
101       1
109       1
150       1


## 3. Inspect representative repeated groups

The examples show only selected fields for readability. Each group matched across all 32 source columns, including the outcome/status fields.

In [3]:
duplicate_examples = duplicate_groups.rename("copies").reset_index()
example_columns = [
    "copies", "hotel", "arrival_date_year", "arrival_date_month",
    "arrival_date_day_of_month", "lead_time", "adults",
    "is_canceled", "reservation_status",
]
print(duplicate_examples[example_columns].head(10).to_string(index=False))

duplicate_row_sample = train.loc[all_duplicate_members, source_columns]
print("\nOutcome/status counts among rows in exact-repeat groups (train only):")
print(duplicate_row_sample[["is_canceled", "reservation_status"]].value_counts().to_string())

 copies      hotel  arrival_date_year arrival_date_month  arrival_date_day_of_month  lead_time  adults  is_canceled reservation_status
    150 City Hotel               2016           February                         17         68       2            1           Canceled
    109 City Hotel               2016               June                         15        188       1            1           Canceled
    101 City Hotel               2016                May                         24        158       1            1           Canceled
    100 City Hotel               2015           December                          8         34       1            1           Canceled
     89 City Hotel               2016               June                         14         71       1            1           Canceled
     72 City Hotel               2015          September                         24        342       2            1           Canceled
     70 City Hotel               2015             Augus

## 4. Compare keep-all with a hypothetical collapse

This is a sensitivity comparison only. The original `train` DataFrame is not modified and no deduplicated file is written.

In [4]:
deduplicated_scenario = train.drop_duplicates(subset=source_columns).copy()

scenario_summary = pd.DataFrame([
    {
        "scenario": "Keep all rows",
        "rows": len(train),
        "cancellation_rate": train["is_canceled"].mean(),
    },
    {
        "scenario": "Hypothetically collapse exact repeats",
        "rows": len(deduplicated_scenario),
        "cancellation_rate": deduplicated_scenario["is_canceled"].mean(),
    },
])
print(scenario_summary.to_string(index=False, formatters={"cancellation_rate": "{:.2%}".format}))

hotel_night_rows = pd.DataFrame({
    "keep_all": train.groupby(["hotel", "arrival_date"]).size(),
    "collapse_exact_repeats": deduplicated_scenario.groupby(["hotel", "arrival_date"]).size(),
}).fillna(0).astype(int)
affected_nights = hotel_night_rows[hotel_night_rows["keep_all"] != hotel_night_rows["collapse_exact_repeats"]]
print(f"\nHotel-arrival nights with a changed booking count in the hypothetical scenario: {len(affected_nights):,}")
print(f"Bookings removed in the hypothetical scenario: {len(train) - len(deduplicated_scenario):,}")

                             scenario  rows cancellation_rate
                        Keep all rows 49157            35.77%
Hypothetically collapse exact repeats 32866            22.17%

Hotel-arrival nights with a changed booking count in the hypothetical scenario: 657
Bookings removed in the hypothetical scenario: 16,291


## 5. Your decision

Evidence does not establish whether identical records are export errors or separate real bookings. Approving “retain” preserves all rows. Disapproving it does not delete anything; first identify a reliable booking key or an evidence-based deduplication rule.

In [5]:
duplicate_decision = "pending"  # Enter "approve" to retain, or "disapprove" to pause and investigate a key/rule.
if duplicate_decision == "approve":
    print("Approved: keep all original training rows.")
elif duplicate_decision == "disapprove":
    print("Disapproved: no rows deleted. First specify a reliable booking key or deduplication rule.")
else:
    print("Pending: the original training rows remain unchanged.")

Pending: the original training rows remain unchanged.


## What this notebook does not decide

It does not determine whether any repeated row is a real booking, remove rows, alter the raw file, inspect choose/proof outcomes, or select a model. Use the evidence above to approve or disapprove the keep-all proposal.